In [ ]:
import requests
import json
import base64
import pandas as pd

In [ ]:
filename = "secret-apikey"

with open(filename, "r") as file:
    WITTENSTEIN_APIKEY = file.read()


AAS_SERVER = "wittenstein-sfh"


if AAS_SERVER == "wittenstein-basyx":
    AAS_SERVER_URL = "https://app-aascore-dev-we-004.azurewebsites.net/"
    proxies = {
    "http": "http://proxy01.wittag.local:9400",
    "https": "http://proxy01.wittag.local:9400"
    }   
elif AAS_SERVER == "wittenstein-sfh":
    AAS_SERVER_URL = "http://z0008a0043.sfh.edge-device.net/faaast/api/v3.0/"
    proxies = {}

custom_headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "X-Api-Key": WITTENSTEIN_APIKEY
}




In [ ]:
# === UPLOAD ===

# Load your AAS JSON for Component A (as built from the template)
with open("AAS-Setup/env_axis.json", "r") as f:   # possibly add encoding: encoding="utf-8"
    aas_payload = json.load(f)


# Upload AAS (POST to /shells):
for aas_json in aas_payload['assetAdministrationShells']:
    print("Uploading AAS:", aas_json.get('idShort', 'N/A'))
    resp = requests.post(
        f"{AAS_SERVER_URL}/shells",
        headers=custom_headers,
        verify=False,
        data=json.dumps(aas_json),
        proxies=proxies
    )
    print("Create AAS status:", resp.status_code, resp.text)



# Upload Submodels:
for submodel_json in aas_payload['submodels']:
    print("Uploading Submodel:", submodel_json.get('idShort', 'N/A'))
    resp = requests.post(
        f"{AAS_SERVER_URL}/submodels",
        headers=custom_headers,
        verify=False,
        data=json.dumps(submodel_json),
        proxies=proxies
    )
    print("Create Submodel status:", resp.status_code, resp.text)

In [ ]:
## alle shells abrufen

url = AAS_SERVER_URL + "shells"

print(url)

try:
    response = requests.get(url, headers=custom_headers, proxies=proxies)
    # Antwort anzeigen
    #print("Status Code:", response.status_code)
    #print("Antwort:", response.text)
    response.raise_for_status()
    filename = "shells_response.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(response.json(), f, indent=2)
    print("Response saved to",filename )
except requests.exceptions.RequestException as e:
    print("Registry connection failed:", e)

In [ ]:
# submodel abrufen Funktion und in datei speichern

def fetch_and_save_submodel(submodel_id, filename):
    
    submodel_id = base64.urlsafe_b64encode(submodel_id.encode("utf-8")).decode("utf-8")
    
    url = f"{AAS_SERVER_URL}submodels/{submodel_id}"
    print(url)
    
    try:
        response = requests.get(url, headers=custom_headers, proxies=proxies)
        print("Status Code:", response.status_code)
        response.raise_for_status()
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(response.json(), f, indent=2)
        print(f"Submodel saved to {filename}")
    except requests.exceptions.RequestException as e:
        print("Connection failed:", e)





In [ ]:
# Healthindex submodel abrufen
fetch_and_save_submodel("https://wgrp.biz/sm/healthindex/1/0/xNA7mRX", "submodel_hi_response.json")

In [ ]:
# Gearbox blob submodel abrufen
fetch_and_save_submodel("https://wgrp.biz/sm/blob/1/0/xNA7mRX", "submodel_gearbox_blob_response.json")


In [ ]:
# motor time series submodel abrufen
fetch_and_save_submodel("urn:https://trumpf.com/aas/submodel/TimeSeries/Motor/0m01/234", "submodel_motor_ts_response.json")

In [ ]:
# value abrufen
submodelID = "urn:https://trumpf.com/aas/submodel/TimeSeries/Motor/0m01/234"
submodelID = base64.urlsafe_b64encode(submodelID.encode("utf-8")).decode("utf-8")

url = AAS_SERVER_URL + "submodels/" + submodelID + "/$value?extent=WithBlobValue"

try:
    response = requests.get(url)
    #print("Connection successful:", response.status_code)
    #print("Response:", response.text)
    response.raise_for_status()
    filename = "submodel_motor_ts_value.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(response.json(), f, indent=2)
    print("Response saved to",filename )
    
except requests.exceptions.RequestException as e:
    print("Connection failed:", e)


In [ ]:
# alle submodels abrufen


url = AAS_SERVER_URL + "submodels"

try:
    response = requests.get(url)
    #print("Connection successful:", response.status_code)
    #print("Response:", response.text)
    response.raise_for_status()
    filename = "submodels_response.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(response.json(), f, indent=2)
    print("Response saved to",filename )
    
except requests.exceptions.RequestException as e:
    print("Connection failed:", e)



## Steuerung der data-source

In [ ]:
#csv hochladen

url = "http://gargamel:8222/fx/upload/csv"
file_path = "dftest.csv"

with open(file_path, "rb") as f:
    files = {"file": (file_path, f)}
    response = requests.post(url, files=files)

print("Status Code:", response.status_code)
print("Response Text:", response.text)


In [ ]:
#TODO
response = requests.get("http://gargamel:8222/fx/transfer/")
print("Status Code:", response.status_code)
print("Response:", response.text)
with open("blob2aas_response.json", "w", encoding="utf-8") as f:
    f.write( response.text)

In [ ]:
#blob eintrag in aas erstellen
response = requests.get("http://gargamel:8222/fx/transfer/blob2aas")
print("Status Code:", response.status_code)
print("Response:", response.text)
with open("blob2aas_response.json", "w", encoding="utf-8") as f:
    f.write( response.text)



In [ ]:

# Original string
original_string = "https://wgrp.biz/sm/blob/1/0/xNA7mRX"

# Convert string to bytes
string_bytes = original_string.encode('utf-8')

# Encode to Base64
base64_bytes = base64.b64encode(string_bytes)

# Convert bytes back to string
base64_string = base64_bytes.decode('utf-8')

print("Base64 Encoded:", base64_string)
